# MDR-TS v7.2
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Tue Jan 6th 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v7.2
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/base/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**

Iterative pruning of all `352` features based on importance. Every iteration, `10%` of them get cut off


For other info see `MDR-TS-v1.0`

## 0. Imports

In [23]:
import os
import json
import math
import random
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML / Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Gradient Boosting (baseline model)
from xgboost import XGBRegressor
import xgboost as xgb

# PyTorch
import torch

# Warnings
import warnings
warnings.filterwarnings("ignore")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

imports loaded
using: cpu


## 1. Environment Setup

In [2]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

# Environment / Runtime Info
def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    # Colab-specific checks
    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

# Plotting defaults
plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.10.18
  NumPy version:  1.26.4
  Pandas version: 2.0.3
  XGBoost version: 2.1.2
  Running in Colab: False
  GPU available: False
environment setup complete


## 2. Data Access

In [3]:
# Project paths
VERSION = "v7"
SUBVERSION = "v7.2"
RUN_NAME = "mdr_ts_v7_2"

PROJECT_ROOT = "/Users/jbalkovec/Desktop/MDR/"
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

# Create output directory if missing
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: /Users/jbalkovec/Desktop/MDR/
  DATA_ROOT:    /Users/jbalkovec/Desktop/MDR//Temporal/Pipeline/data
  SPLIT_ROOT:   /Users/jbalkovec/Desktop/MDR//Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  /Users/jbalkovec/Desktop/MDR//Models/Temporal/v7/v7.2

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


## 3. Data Loading

In [5]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_new/train_derived_new.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_new/val_derived_new.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_new/test_derived_new.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

DROP_COLS = ["slope", "elev"]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    cols_present = [c for c in DROP_COLS if c in d.columns]
    d.drop(columns=cols_present, inplace=True)
    print(f"{name}: dropped columns {cols_present}")
    print(f"\n{name}: shape={d.shape}")
    print(f"{name}: columns={len(d.columns)}")

Split files:
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/train_derived_new.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/val_derived_new.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/test_derived_new.csv
train: dropped columns ['slope', 'elev']

train: shape=(16972, 352)
train: columns=352
val: dropped columns ['slope', 'elev']

val: shape=(2919, 352)
val: columns=352
test: dropped columns ['slope', 'elev']

test: shape=(2829, 352)
test: columns=352


In [8]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: 352

columns:
['station_id', 'date', 'longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'aspect', 'DOY', 'soil_moisture_5cm', 'F_NDVI', 'F_NDMI', 'F_MSI', 'E_SAR_ratio', 'E_SAR_diff', 'G_API', 'G_DSLR', 'G_rain_sum_3d', 'G_rain_sum_7d', 'G_rain_sum_30d', 'A_d_G_API_kobs1', 'A_d_G_API_kobs2', 'A_d_G_API_kobs5', 'A_d_G_API_kobs7', 'A_d_G_API_kobs14']


## 4. Data Sanity Checks

In [32]:
TARGET_COL = "soil_moisture_5cm"

KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS = ['precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8',
                's2_b11', 's2_b12', 'LST_modis', 'aspect', 'DOY',
                'F_NDVI', 'F_NDMI', 'F_MSI', 'E_SAR_ratio',
                'E_SAR_diff', 'G_API', 'G_DSLR', 'G_rain_sum_3d',
                'G_rain_sum_7d', 'G_rain_sum_30d', 'A_d_G_API_kobs1',
                'A_d_G_API_kobs2', 'A_d_G_API_kobs5', 'A_d_G_API_kobs7',
                'A_d_G_API_kobs14', 'A_d_G_API_kobs30', 'A_grad_G_API_kobs7',
                'A_grad_G_API_kobs14', 'A_grad_G_API_kobs30', 'A_pct_G_API',
                'V_rollstd_G_API_kobs7', 'V_rollrng_G_API_kobs7',
                'V_rollcv_G_API_kobs7', 'V_rollmean_G_API_kobs7',
                'V_rollmin_G_API_kobs7', 'V_rollmax_G_API_kobs7',
                'V_ema_G_API_kobs7', 'V_rollstd_G_API_kobs14',
                'V_rollrng_G_API_kobs14', 'V_rollcv_G_API_kobs14',
                'V_rollmean_G_API_kobs14', 'V_rollmin_G_API_kobs14',
                'V_rollmax_G_API_kobs14', 'V_ema_G_API_kobs14',
                'V_rollstd_G_API_kobs30', 'V_rollrng_G_API_kobs30',
                'V_rollcv_G_API_kobs30', 'V_rollmean_G_API_kobs30',
                'V_rollmin_G_API_kobs30', 'V_rollmax_G_API_kobs30',
                'V_ema_G_API_kobs30', 'C_lag_G_API_kobs1', 'C_lag_G_API_kobs2',
                'C_lag_G_API_kobs5', 'C_lag_G_API_kobs6', 'C_lag_G_API_kobs12',
                'C_lag_G_API_kobs30', 'C_smm_G_API_alpha0.85_n5', 'A_d_F_NDMI_kobs1',
                'A_d_F_NDMI_kobs2', 'A_d_F_NDMI_kobs5', 'A_d_F_NDMI_kobs7',
                'A_d_F_NDMI_kobs14', 'A_d_F_NDMI_kobs30', 'A_grad_F_NDMI_kobs7',
                'A_grad_F_NDMI_kobs14', 'A_grad_F_NDMI_kobs30', 'A_pct_F_NDMI',
                'V_rollstd_F_NDMI_kobs7', 'V_rollrng_F_NDMI_kobs7',
                'V_rollcv_F_NDMI_kobs7', 'V_rollmean_F_NDMI_kobs7',
                'V_rollmin_F_NDMI_kobs7', 'V_rollmax_F_NDMI_kobs7',
                'V_ema_F_NDMI_kobs7', 'V_rollstd_F_NDMI_kobs14',
                'V_rollrng_F_NDMI_kobs14', 'V_rollcv_F_NDMI_kobs14',
                'V_rollmean_F_NDMI_kobs14', 'V_rollmin_F_NDMI_kobs14',
                'V_rollmax_F_NDMI_kobs14', 'V_ema_F_NDMI_kobs14',
                'V_rollstd_F_NDMI_kobs30', 'V_rollrng_F_NDMI_kobs30',
                'V_rollcv_F_NDMI_kobs30', 'V_rollmean_F_NDMI_kobs30',
                'V_rollmin_F_NDMI_kobs30', 'V_rollmax_F_NDMI_kobs30',
                'V_ema_F_NDMI_kobs30', 'C_lag_F_NDMI_kobs1', 'C_lag_F_NDMI_kobs2',
                'C_lag_F_NDMI_kobs5', 'C_lag_F_NDMI_kobs6', 'C_lag_F_NDMI_kobs12',
                'C_lag_F_NDMI_kobs30', 'C_smm_F_NDMI_alpha0.85_n5', 'A_d_E_SAR_ratio_kobs1',
                'A_d_E_SAR_ratio_kobs2', 'A_d_E_SAR_ratio_kobs5', 'A_d_E_SAR_ratio_kobs7',
                'A_d_E_SAR_ratio_kobs14', 'A_d_E_SAR_ratio_kobs30', 'A_grad_E_SAR_ratio_kobs7',
                'A_grad_E_SAR_ratio_kobs14', 'A_grad_E_SAR_ratio_kobs30', 'A_pct_E_SAR_ratio',
                'V_rollstd_E_SAR_ratio_kobs7', 'V_rollrng_E_SAR_ratio_kobs7',
                'V_rollcv_E_SAR_ratio_kobs7', 'V_rollmean_E_SAR_ratio_kobs7',
                'V_rollmin_E_SAR_ratio_kobs7', 'V_rollmax_E_SAR_ratio_kobs7',
                'V_ema_E_SAR_ratio_kobs7', 'V_rollstd_E_SAR_ratio_kobs14',
                'V_rollrng_E_SAR_ratio_kobs14', 'V_rollcv_E_SAR_ratio_kobs14',
                'V_rollmean_E_SAR_ratio_kobs14', 'V_rollmin_E_SAR_ratio_kobs14',
                'V_rollmax_E_SAR_ratio_kobs14', 'V_ema_E_SAR_ratio_kobs14', 'V_rollstd_E_SAR_ratio_kobs30',
                'V_rollrng_E_SAR_ratio_kobs30', 'V_rollcv_E_SAR_ratio_kobs30', 'V_rollmean_E_SAR_ratio_kobs30',
                'V_rollmin_E_SAR_ratio_kobs30', 'V_rollmax_E_SAR_ratio_kobs30', 'V_ema_E_SAR_ratio_kobs30',
                'C_lag_E_SAR_ratio_kobs1', 'C_lag_E_SAR_ratio_kobs2', 'C_lag_E_SAR_ratio_kobs5',
                'C_lag_E_SAR_ratio_kobs6', 'C_lag_E_SAR_ratio_kobs12', 'C_lag_E_SAR_ratio_kobs30',
                'C_smm_E_SAR_ratio_alpha0.85_n5', 'A_d_LST_modis_kobs1', 'A_d_LST_modis_kobs2',
                'A_d_LST_modis_kobs5', 'A_d_LST_modis_kobs7', 'A_d_LST_modis_kobs14', 'A_d_LST_modis_kobs30',
                'A_grad_LST_modis_kobs7', 'A_grad_LST_modis_kobs14', 'A_grad_LST_modis_kobs30',
                'A_pct_LST_modis', 'V_rollstd_LST_modis_kobs7', 'V_rollrng_LST_modis_kobs7',
                'V_rollcv_LST_modis_kobs7', 'V_rollmean_LST_modis_kobs7', 'V_rollmin_LST_modis_kobs7',
                'V_rollmax_LST_modis_kobs7', 'V_ema_LST_modis_kobs7', 'V_rollstd_LST_modis_kobs14',
                'V_rollrng_LST_modis_kobs14', 'V_rollcv_LST_modis_kobs14', 'V_rollmean_LST_modis_kobs14',
                'V_rollmin_LST_modis_kobs14', 'V_rollmax_LST_modis_kobs14', 'V_ema_LST_modis_kobs14',
                'V_rollstd_LST_modis_kobs30', 'V_rollrng_LST_modis_kobs30', 'V_rollcv_LST_modis_kobs30',
                'V_rollmean_LST_modis_kobs30', 'V_rollmin_LST_modis_kobs30', 'V_rollmax_LST_modis_kobs30',
                'V_ema_LST_modis_kobs30', 'C_lag_LST_modis_kobs1', 'C_lag_LST_modis_kobs2',
                'C_lag_LST_modis_kobs5', 'C_lag_LST_modis_kobs6', 'C_lag_LST_modis_kobs12',
                'C_lag_LST_modis_kobs30', 'C_smm_LST_modis_alpha0.85_n5', 'A_d_F_NDVI_kobs1',
                'A_d_F_NDVI_kobs2', 'A_d_F_NDVI_kobs5', 'A_d_F_NDVI_kobs7', 'A_d_F_NDVI_kobs14',
                'A_d_F_NDVI_kobs30', 'A_grad_F_NDVI_kobs7', 'A_grad_F_NDVI_kobs14',
                'A_grad_F_NDVI_kobs30', 'A_pct_F_NDVI', 'V_rollstd_F_NDVI_kobs7',
                'V_rollrng_F_NDVI_kobs7', 'V_rollcv_F_NDVI_kobs7', 'V_rollmean_F_NDVI_kobs7',
                'V_rollmin_F_NDVI_kobs7', 'V_rollmax_F_NDVI_kobs7', 'V_ema_F_NDVI_kobs7',
                'V_rollstd_F_NDVI_kobs14', 'V_rollrng_F_NDVI_kobs14', 'V_rollcv_F_NDVI_kobs14',
                'V_rollmean_F_NDVI_kobs14', 'V_rollmin_F_NDVI_kobs14', 'V_rollmax_F_NDVI_kobs14',
                'V_ema_F_NDVI_kobs14', 'V_rollstd_F_NDVI_kobs30', 'V_rollrng_F_NDVI_kobs30',
                'V_rollcv_F_NDVI_kobs30', 'V_rollmean_F_NDVI_kobs30', 'V_rollmin_F_NDVI_kobs30',
                'V_rollmax_F_NDVI_kobs30', 'V_ema_F_NDVI_kobs30', 'C_lag_F_NDVI_kobs1',
                'C_lag_F_NDVI_kobs2', 'C_lag_F_NDVI_kobs5', 'C_lag_F_NDVI_kobs6', 'C_lag_F_NDVI_kobs12',
                'C_lag_F_NDVI_kobs30', 'C_smm_F_NDVI_alpha0.85_n5', 'A_d_E_SAR_diff_kobs1',
                'A_d_E_SAR_diff_kobs2', 'A_d_E_SAR_diff_kobs5', 'A_d_E_SAR_diff_kobs7',
                'A_d_E_SAR_diff_kobs14', 'A_d_E_SAR_diff_kobs30', 'A_grad_E_SAR_diff_kobs7',
                'A_grad_E_SAR_diff_kobs14', 'A_grad_E_SAR_diff_kobs30', 'A_pct_E_SAR_diff',
                'V_rollstd_E_SAR_diff_kobs7', 'V_rollrng_E_SAR_diff_kobs7', 'V_rollcv_E_SAR_diff_kobs7',
                'V_rollmean_E_SAR_diff_kobs7', 'V_rollmin_E_SAR_diff_kobs7', 'V_rollmax_E_SAR_diff_kobs7',
                'V_ema_E_SAR_diff_kobs7', 'V_rollstd_E_SAR_diff_kobs14', 'V_rollrng_E_SAR_diff_kobs14',
                'V_rollcv_E_SAR_diff_kobs14', 'V_rollmean_E_SAR_diff_kobs14', 'V_rollmin_E_SAR_diff_kobs14',
                'V_rollmax_E_SAR_diff_kobs14', 'V_ema_E_SAR_diff_kobs14', 'V_rollstd_E_SAR_diff_kobs30',
                'V_rollrng_E_SAR_diff_kobs30', 'V_rollcv_E_SAR_diff_kobs30', 'V_rollmean_E_SAR_diff_kobs30',
                'V_rollmin_E_SAR_diff_kobs30', 'V_rollmax_E_SAR_diff_kobs30', 'V_ema_E_SAR_diff_kobs30',
                'C_lag_E_SAR_diff_kobs1', 'C_lag_E_SAR_diff_kobs2', 'C_lag_E_SAR_diff_kobs5', 'C_lag_E_SAR_diff_kobs6',
                'C_lag_E_SAR_diff_kobs12', 'C_lag_E_SAR_diff_kobs30', 'C_smm_E_SAR_diff_alpha0.85_n5', 'A_d_s2_b11_kobs1',
                'A_d_s2_b11_kobs2', 'A_d_s2_b11_kobs5', 'A_d_s2_b11_kobs7', 'A_d_s2_b11_kobs14',
                'A_d_s2_b11_kobs30', 'A_grad_s2_b11_kobs7', 'A_grad_s2_b11_kobs14', 'A_grad_s2_b11_kobs30',
                'A_pct_s2_b11', 'V_rollstd_s2_b11_kobs7', 'V_rollrng_s2_b11_kobs7', 'V_rollcv_s2_b11_kobs7',
                'V_rollmean_s2_b11_kobs7', 'V_rollmin_s2_b11_kobs7', 'V_rollmax_s2_b11_kobs7', 'V_ema_s2_b11_kobs7',
                'V_rollstd_s2_b11_kobs14', 'V_rollrng_s2_b11_kobs14', 'V_rollcv_s2_b11_kobs14',
                'V_rollmean_s2_b11_kobs14', 'V_rollmin_s2_b11_kobs14', 'V_rollmax_s2_b11_kobs14', 'V_ema_s2_b11_kobs14',
                'V_rollstd_s2_b11_kobs30', 'V_rollrng_s2_b11_kobs30', 'V_rollcv_s2_b11_kobs30',
                'V_rollmean_s2_b11_kobs30', 'V_rollmin_s2_b11_kobs30', 'V_rollmax_s2_b11_kobs30',
                'V_ema_s2_b11_kobs30', 'C_lag_s2_b11_kobs1', 'C_lag_s2_b11_kobs2', 'C_lag_s2_b11_kobs5',
                'C_lag_s2_b11_kobs6', 'C_lag_s2_b11_kobs12', 'C_lag_s2_b11_kobs30', 'C_smm_s2_b11_alpha0.85_n5',
                'A_d_s2_b12_kobs1', 'A_d_s2_b12_kobs2', 'A_d_s2_b12_kobs5', 'A_d_s2_b12_kobs7', 'A_d_s2_b12_kobs14',
                'A_d_s2_b12_kobs30', 'A_grad_s2_b12_kobs7', 'A_grad_s2_b12_kobs14', 'A_grad_s2_b12_kobs30',
                'A_pct_s2_b12', 'V_rollstd_s2_b12_kobs7', 'V_rollrng_s2_b12_kobs7', 'V_rollcv_s2_b12_kobs7',
                'V_rollmean_s2_b12_kobs7', 'V_rollmin_s2_b12_kobs7', 'V_rollmax_s2_b12_kobs7',
                'V_ema_s2_b12_kobs7', 'V_rollstd_s2_b12_kobs14', 'V_rollrng_s2_b12_kobs14',
                'V_rollcv_s2_b12_kobs14', 'V_rollmean_s2_b12_kobs14', 'V_rollmin_s2_b12_kobs14',
                'V_rollmax_s2_b12_kobs14', 'V_ema_s2_b12_kobs14', 'V_rollstd_s2_b12_kobs30',
                'V_rollrng_s2_b12_kobs30', 'V_rollcv_s2_b12_kobs30', 'V_rollmean_s2_b12_kobs30',
                'V_rollmin_s2_b12_kobs30', 'V_rollmax_s2_b12_kobs30', 'V_ema_s2_b12_kobs30',
                'C_lag_s2_b12_kobs1', 'C_lag_s2_b12_kobs2', 'C_lag_s2_b12_kobs5', 'C_lag_s2_b12_kobs6',
                'C_lag_s2_b12_kobs12', 'C_lag_s2_b12_kobs30', 'C_smm_s2_b12_alpha0.85_n5', 'E_dVV_1',
                'E_rough_s1_vv_kobs7', 'E_rough_s1_vh_kobs7', 'E_rough_s1_vv_kobs14', 'E_rough_s1_vh_kobs14',
                'I_ts_spike_s1_vv','D_sa_F_NDMI', 'D_z_F_NDMI',
                'D_sa_E_SAR_ratio', 'D_z_E_SAR_ratio', 'D_sa_LST_modis', 'D_z_LST_modis', 'D_fft_dom_F_NDMI_kobs30',
                'D_fft_ent_F_NDMI_kobs30', 'D_fft_dom_E_SAR_ratio_kobs30', 'D_fft_ent_E_SAR_ratio_kobs30',
                'D_fft_dom_LST_modis_kobs30', 'D_fft_ent_LST_modis_kobs30', 'G_DSLR_isnan']

# quick validation
expected = set(KEEP_META_COLS + FEATURE_COLS + [TARGET_COL])
missing_train = sorted(list(expected - set(train_df.columns)))
missing_val   = sorted(list(expected - set(val_df.columns)))
missing_test  = sorted(list(expected - set(test_df.columns)))

if missing_train or missing_val or missing_test:
    raise ValueError(
        f"Missing columns:\n"
        f"  train: {missing_train}\n"
        f"  val:   {missing_val}\n"
        f"  test:  {missing_test}"
    )

print("Columns locked")
print("  Features:", len(FEATURE_COLS))
print("  Target:  ", TARGET_COL)

Columns locked
  Features: 343
  Target:   soil_moisture_5cm


## 5. Train / Validation / Test Split

### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [33]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")


=== SPLIT SUMMARY ===

TRAIN
  rows:     16972
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
  date range: 2011-10-06 00:00:00 -- 2021-09-23 00:00:00

VAL
  rows:     2919
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2021-09-24 00:00:00 -- 2023-11-12 00:00:00

TEST
  rows:     2829
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
  date range: 2023-11-13 00:00:00 -- 2025-12-31 00:00:00

=== LEAKAGE CHECK ===
train ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
val   ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']

-- split locked --


## 6. Model Definition

This section defines the MDR-TS v1.0 baseline model.
The emphasis here is **transparency and debuggability**, not maximal performance.

The model is intentionally simple and interpretable:
- Explicit feature matrices
- No feature embedding or deep architecture
- Full access to feature importance and diagnostics

This serves as a reference point for all future MDR models.

### 6.1 Feature Matrix Construction

In [34]:
X_train = train_df[FEATURE_COLS].copy()
y_train = train_df[TARGET_COL].copy()

X_val = val_df[FEATURE_COLS].copy()
y_val = val_df[TARGET_COL].copy()

X_test = test_df[FEATURE_COLS].copy()
y_test = test_df[TARGET_COL].copy()

print("Feature matrix shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}")

Feature matrix shapes:
  X_train: (16972, 343)
  X_val:   (2919, 343)
  X_test:  (2829, 343)
  y_train: (16972,)


In [35]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def metrics_row(step, n_features, removed, best_iteration, feats, y_tr, y_va, y_te, p_tr, p_va, p_te, top10):
    return {
        "step": step,
        "n_features": n_features,
        "removed": removed,
        "best_iteration": best_iteration,
        "r2_train": float(r2_score(y_tr, p_tr)),
        "r2_val":   float(r2_score(y_va, p_va)),
        "r2_test":  float(r2_score(y_te, p_te)),
        "rmse_train": rmse(y_tr, p_tr),
        "rmse_val":   rmse(y_va, p_va),
        "rmse_test":  rmse(y_te, p_te),
        "mae_train": float(mean_absolute_error(y_tr, p_tr)),
        "mae_val":   float(mean_absolute_error(y_va, p_va)),
        "mae_test":  float(mean_absolute_error(y_te, p_te)),
        "top10_gain": ", ".join([f"{k}:{v:.3g}" for k, v in top10]),
    }

In [41]:
def _perm_importance_r2(model, X_val, y_val, feats, n_rows=4000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(X_val)
    idx = np.arange(n)
    if n > n_rows:
        idx = rng.choice(idx, size=n_rows, replace=False)

    Xb = X_val.loc[X_val.index[idx], feats].copy()
    yb = y_val.loc[y_val.index[idx]]

    base = float(r2_score(yb, model.predict(Xb)))

    imps = {}
    for c in feats:
        x_save = Xb[c].to_numpy().copy()
        rng.shuffle(Xb[c].to_numpy())  # in-place shuffle view
        score = float(r2_score(yb, model.predict(Xb)))
        imps[c] = base - score
        Xb[c] = x_save  # restore

    return pd.Series(imps).sort_values(ascending=False)


In [42]:
def iterative_prune_bottom_10pct_no_es(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    feature_cols,
    model_config,
    *,
    prune_frac=0.10,
    min_features=25,
    verbose_eval=False,
    use_perm_fallback=True,
    perm_rows=4000,
    perm_seed=0,
):
    feats = list(feature_cols)
    history = []
    step = 0

    while len(feats) >= min_features:
        step += 1

        cfg = dict(model_config)
        cfg["n_estimators"] = int(cfg.get("n_estimators", 1000))

        model = XGBRegressor(**cfg)
        model.fit(X_train[feats], y_train, verbose=verbose_eval)

        p_tr = model.predict(X_train[feats])
        p_va = model.predict(X_val[feats])
        p_te = model.predict(X_test[feats])

        r2_tr = float(r2_score(y_train, p_tr))
        r2_va = float(r2_score(y_val, p_va))
        r2_te = float(r2_score(y_test, p_te))

        # --- Importance used for pruning ---
        imp = pd.Series(model.feature_importances_, index=feats).astype(float)

        # If importances are all zero-ish, fall back to permutation
        if use_perm_fallback and (imp.sum() == 0.0 or (imp > 0).sum() == 0):
            imp_used = _perm_importance_r2(
                model, X_val, y_val, feats,
                n_rows=perm_rows, seed=perm_seed + step
            )
            imp_source = "perm_val_r2"
        else:
            imp_used = imp.sort_values(ascending=False)
            imp_source = "feature_importances_"

        top10 = list(imp_used.head(10).items())

        # prune bottom X%
        prune_n = int(np.ceil(len(feats) * prune_frac))
        prune_n = max(1, prune_n)
        prune_n = min(prune_n, max(0, len(feats) - min_features))

        to_remove = imp_used.sort_values(ascending=True).head(prune_n).index.tolist()

        history.append({
            "step": step,
            "n_features": len(feats),
            "removed": prune_n,
            "importance_source": imp_source,
            "r2_train": r2_tr,
            "r2_val": r2_va,
            "r2_test": r2_te,
            "rmse_train": rmse(y_train, p_tr),
            "rmse_val": rmse(y_val, p_va),
            "rmse_test": rmse(y_test, p_te),
            "mae_train": float(mean_absolute_error(y_train, p_tr)),
            "mae_val": float(mean_absolute_error(y_val, p_va)),
            "mae_test": float(mean_absolute_error(y_test, p_te)),
            "top10_imp": ", ".join([f"{k}:{v:.3g}" for k, v in top10]),
        })

        print(
            f"[step {step:02d}] n={len(feats):3d}  "
            f"R2 train/val/test = {r2_tr:.4f} / {r2_va:.4f} / {r2_te:.4f}  "
            f"(remove {prune_n})  via {imp_source}"
        )

        if prune_n == 0:
            break

        feats = [f for f in feats if f not in set(to_remove)]

    return pd.DataFrame(history)

In [43]:
MODEL_CONFIG_WIDE = {
    "n_estimators": 1500,          # no early stopping in your setup, so keep sane
    "max_depth": 3,                # key change: reduce specialization
    "learning_rate": 0.03,
    "min_child_weight": 20,        # stronger regularization on splits
    "gamma": 2.0,                  # require meaningful gain to split
    "subsample": 0.70,
    "colsample_bytree": 0.60,      # force feature diversity, helps with wide sets
    "reg_alpha": 0.10,
    "reg_lambda": 8.0,
    "objective": "reg:squarederror",
    "random_state": SEED,
    "n_jobs": -1,
}

MODEL_CONFIG_PRUNE = dict(MODEL_CONFIG_WIDE)
MODEL_CONFIG_PRUNE["n_estimators"] = 1200  # since you can't early stop

results_prune = iterative_prune_bottom_10pct_no_es(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    feature_cols=FEATURE_COLS,
    model_config=MODEL_CONFIG_PRUNE,
    prune_frac=0.10,
    min_features=25,
    verbose_eval=False,
    use_perm_fallback=True,   # keep this on
    perm_rows=4000,           # drop to 2000 if slow
)

[step 01] n=343  R2 train/val/test = 0.7284 / 0.7267 / 0.6983  (remove 35)  via feature_importances_
[step 02] n=308  R2 train/val/test = 0.7283 / 0.7282 / 0.6976  (remove 31)  via feature_importances_
[step 03] n=277  R2 train/val/test = 0.7290 / 0.7344 / 0.7021  (remove 28)  via feature_importances_
[step 04] n=249  R2 train/val/test = 0.7269 / 0.7265 / 0.7051  (remove 25)  via feature_importances_
[step 05] n=224  R2 train/val/test = 0.7254 / 0.7269 / 0.7040  (remove 23)  via feature_importances_
[step 06] n=201  R2 train/val/test = 0.7286 / 0.7354 / 0.7060  (remove 21)  via feature_importances_
[step 07] n=180  R2 train/val/test = 0.7275 / 0.7298 / 0.6967  (remove 18)  via feature_importances_
[step 08] n=162  R2 train/val/test = 0.7258 / 0.7241 / 0.7013  (remove 17)  via feature_importances_
[step 09] n=145  R2 train/val/test = 0.7274 / 0.7235 / 0.6968  (remove 15)  via feature_importances_
[step 10] n=130  R2 train/val/test = 0.7292 / 0.7269 / 0.7021  (remove 13)  via feature_imp

In [40]:
results_prune

,step,n_features,removed,r2_train,r2_val,r2_test,rmse_train,rmse_val,rmse_test,mae_train,mae_val,mae_test,top10_gain
0,1,343,35,0.728365,0.726691,0.698289,0.053293,0.052654,0.051374,0.043983,0.042708,0.042837,"precip_mm:0, V_ema_E_SAR_diff_kobs7:0, V_rolls..."
1,2,308,31,0.697149,0.684400,0.653864,0.056272,0.056582,0.055026,0.046672,0.045833,0.046262,"s1_vv:0, V_rollmin_E_SAR_diff_kobs30:0, C_lag_..."
2,3,277,28,0.669793,0.672657,0.657263,0.058759,0.057625,0.054756,0.048625,0.047507,0.045941,"A_d_G_API_kobs5:0, A_d_s2_b11_kobs2:0, A_grad_..."
3,4,249,25,0.651474,0.636536,0.627940,0.060367,0.060721,0.057050,0.050253,0.050366,0.048219,"C_lag_G_API_kobs5:0, V_rollstd_s2_b11_kobs14:0..."
4,5,224,23,0.605569,0.573371,0.582125,0.064219,0.065786,0.060460,0.052406,0.055292,0.051107,"V_ema_F_NDMI_kobs14:0, V_rollmin_E_SAR_ratio_k..."
5,6,201,21,0.600635,0.581254,0.591608,0.064620,0.065175,0.059771,0.052776,0.054793,0.050649,"V_rollmin_E_SAR_ratio_kobs14:0, C_lag_s2_b11_k..."
6,7,180,18,0.596590,0.574761,0.593647,0.064946,0.065679,0.059621,0.053026,0.055340,0.050599,"C_lag_E_SAR_ratio_kobs30:0, V_rollmin_s2_b11_k..."
7,8,162,17,0.383532,0.300051,0.357432,0.080285,0.084264,0.074973,0.066681,0.072028,0.064231,"V_rollcv_LST_modis_kobs30:0, V_ema_s2_b12_kobs..."
8,9,145,15,0.357903,0.268474,0.361987,0.081937,0.086143,0.074707,0.068126,0.073951,0.063307,"A_d_F_NDVI_kobs1:0, V_ema_s2_b11_kobs14:0, A_d..."
9,10,130,13,0.351558,0.261885,0.354449,0.082341,0.086531,0.075147,0.068439,0.074225,0.063851,"V_ema_F_NDVI_kobs7:0, V_rollstd_s2_b12_kobs30:..."


In [44]:
results_prune[["step","n_features","removed","importance_source","r2_train","r2_val","r2_test"]].head(10)

,step,n_features,removed,importance_source,r2_train,r2_val,r2_test
0,1,343,35,feature_importances_,0.728365,0.726691,0.698289
1,2,308,31,feature_importances_,0.728314,0.728170,0.697620
2,3,277,28,feature_importances_,0.729050,0.734440,0.702128
3,4,249,25,feature_importances_,0.726875,0.726539,0.705104
4,5,224,23,feature_importances_,0.725391,0.726914,0.704048
5,6,201,21,feature_importances_,0.728570,0.735362,0.706009
6,7,180,18,feature_importances_,0.727542,0.729839,0.696712
7,8,162,17,feature_importances_,0.725760,0.724093,0.701269
8,9,145,15,feature_importances_,0.727418,0.723515,0.696774
9,10,130,13,feature_importances_,0.729237,0.726876,0.702140


In [45]:
results_prune.sort_values("r2_test", ascending=False).head(5)

,step,n_features,removed,importance_source,r2_train,r2_val,r2_test,rmse_train,rmse_val,rmse_test,mae_train,mae_val,mae_test,top10_imp
5,6,201,21,feature_importances_,0.728570,0.735362,0.706009,0.053273,0.051812,0.050713,0.044013,0.042176,0.042257,"V_rollmin_G_API_kobs7:0.0954, V_rollmin_LST_mo..."
3,4,249,25,feature_importances_,0.726875,0.726539,0.705104,0.053439,0.052669,0.050791,0.044103,0.042709,0.042323,"V_ema_G_API_kobs14:0.102, V_ema_LST_modis_kobs..."
4,5,224,23,feature_importances_,0.725391,0.726914,0.704048,0.053584,0.052633,0.050881,0.044260,0.042687,0.042382,"V_rollmin_LST_modis_kobs30:0.0985, V_rollmin_G..."
15,16,67,7,feature_importances_,0.726368,0.726422,0.703891,0.053489,0.052680,0.050895,0.044184,0.042726,0.042336,"V_rollmean_LST_modis_kobs30:0.103, V_ema_LST_m..."
9,10,130,13,feature_importances_,0.729237,0.726876,0.702140,0.053208,0.052637,0.051045,0.043891,0.042775,0.042460,"V_rollmin_LST_modis_kobs30:0.105, V_rollmin_G_..."


---

_Jakob Balkovec_